In [1]:
import pickle

In [2]:
with open('/root/autodl-tmp/chuandian_eq/data/taxi/raw/train.pkl', 'rb') as f:
    data = pickle.load(f)

In [3]:
print(data['train'][0])
print(len(data['train']))

[{'idx_event': 1, 'type_event': 8, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 3, 'time_since_start': 0.07888888888888888, 'time_since_last_event': 0.07888888888888888}, {'idx_event': 3, 'type_event': 8, 'time_since_start': 0.27666666666666667, 'time_since_last_event': 0.19777777777777777}, {'idx_event': 4, 'type_event': 3, 'time_since_start': 0.37972222222222224, 'time_since_last_event': 0.10305555555555557}, {'idx_event': 5, 'type_event': 8, 'time_since_start': 0.5475, 'time_since_last_event': 0.16777777777777775}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 1.0013888888888889, 'time_since_last_event': 0.4538888888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 1.4066666666666667, 'time_since_last_event': 0.40527777777777785}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 1.8002777777777779, 'time_since_last_event': 0.39361111111111113}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 1.8716666666666666

In [4]:
from src.data.sequence import Sequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    t_start = 0.0
    t_nll_start = 0.0
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return Sequence(
        t_start=t_start,
        t_nll_start=t_nll_start,
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["train"]]

/root/autodl-tmp/chuandian_eq/src/data/sequence.py:153: UserWarning: Found 1 zero inter-event times in the sequence. This violates fundamental assumptions of TPP models and may lead to incorrect log-likelihood values.
  warnings.warn(


In [6]:
from src.data.batch import Batch


In [7]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [8]:
for batch in loader:
    print(batch.keys())
    break

['inter_times', 'arrival_times', 't_start', 't_end', 't_nll_start', 'nll_mask', 'start_idx', 'end_idx', 'non_pad_mask', 'type_seq', 'type_event']


In [9]:
batch.type_event

tensor([[8, 3, 8,  ..., 3, 0, 0],
        [8, 3, 8,  ..., 1, 0, 0],
        [5, 3, 8,  ..., 0, 0, 0],
        ...,
        [8, 3, 8,  ..., 3, 8, 3],
        [8, 3, 8,  ..., 3, 0, 0],
        [5, 0, 5,  ..., 3, 5, 0]])

In [10]:
batch.type_seq

tensor([[-100,    3,    8,  ...,    3, -100, -100],
        [-100,    3,    8,  ...,    1, -100, -100],
        [-100,    3,    8,  ...,    0, -100, -100],
        ...,
        [-100,    3,    8,  ...,    3,    8,    3],
        [-100,    3,    8,  ...,    3, -100, -100],
        [-100,    0,    5,  ...,    3,    5,    0]])